# Laboratorio 7 - Deep Learning

- Juan Solís - 23720
- Victor Pérez - 23731
- Diego Flores - 23714

## Task 1

### Task 1.1

Se descarga el dataset ETTh1 y se construye el pipeline de preprocesamiento sobre la variable objetivo `OT` (Oil Temperature). Se normaliza con z-score guardando `mu` y `sigma`, se divide en bloques temporales contiguos 60/20/20 (train/val/test) y se generan ventanas deslizantes de longitud $L = 96$ para los horizontes $H \in \{24, 48\}$, donde $X^{(t)} = (x_{t-L+1}, \dots, x_t)$ y $y^{(t)} = x_{t+H}$.

In [7]:
import urllib.request
import numpy as np
import torch

URL = ("https://raw.githubusercontent.com/zhouhaoyi/"
       "ETDataset/main/ETT-small/ETTh1.csv")
urllib.request.urlretrieve(URL, "ETTh1.csv")

data = np.genfromtxt("ETTh1.csv", delimiter=",", skip_header=1)
OT  = data[:, -1].astype(np.float32)   # Oil Temperature: columna objetivo
ALL = data[:, 1:].astype(np.float32)   # 7 variables (6 de carga + OT)

# a. Normalizacion z-score de OT
mu, sigma = OT.mean(), OT.std()
OT_norm =torch.tensor((OT - mu) / sigma)

# b. Split temporal contiguo 60/20/20
n =len(OT_norm)
n_tr, n_vl =int(0.6 * n), int(0.8 * n)
OT_train, OT_val, OT_test = OT_norm[:n_tr], OT_norm[n_tr:n_vl],OT_norm[n_vl:]

# c. Ventanas deslizantes
def make_windows( series, L, H ):
    N = len(series) - L - H + 1
    X = torch.stack([series[i:i + L] for i in range(N)])
    y = series[L - 1 + H : L - 1 + H + N]
    return X, y

L=96
X_tr_24, y_tr_24 = make_windows(OT_train, L, 24)
X_vl_24, y_vl_24 = make_windows(OT_val, L, 24)
X_te_24, y_te_24 = make_windows(OT_test, L, 24)
X_tr_48, y_tr_48 = make_windows(OT_train, L, 48)
X_vl_48, y_vl_48 = make_windows(OT_val, L, 48)
X_te_48, y_te_48 = make_windows(OT_test, L, 48)

print(f"mu={mu:.4f}, sigma={sigma:.4f}")
print(f"Train: {len(OT_train)}, Val: {len(OT_val)}, Test: {len(OT_test)}")
print(f"X_tr_24: {tuple(X_tr_24.shape)}, X_tr_48: {tuple(X_tr_48.shape)}")

mu=13.3247, sigma=8.5667
Train: 10452, Val: 3484, Test: 3484
X_tr_24: (10333, 96), X_tr_48: (10309, 96)
